In [ ]:
import os
import json
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy.stats import spearmanr
from openai import OpenAI, APIConnectionError, APIStatusError
from abc import ABC, abstractmethod
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path
import time
from time import perf_counter
from concurrent.futures import ThreadPoolExecutor, as_completed

pd.set_option("display.max_colwidth", 110)

In [ ]:
# PROMPT TEMPLATES

FEWSHOT_PROMPT_DE = """
Annotate a pair of English-German input sentences. The goal is to label every word in each sentence regarding its semantic similarity to the words in the other sentence.

## Example 1:
Input sentence 1: ["Former Nazi death camp guard Demjanjuk dead at 91"]
Input sentence 2: ["John Demjanjuk, verurteilter Nazi-Todeslagerwächter, stirbt im Alter von 91 Jahren"]
Output: {"sentence1": [["Former", 1],["Nazi", 5],["death", 5],["camp", 5],["guard", 5],["Demjanjuk", 5],["dead", 5],["at", 5],["91", 5]],"sentence2": [["John", 0],["Demjanjuk", 5],[",", -1],["verurteilter", 1],["Nazi", 1],["-", -1],["Todeslagerwächter", 5],[",", -1],["stirbt", 5],["im", 4],["Alter", 4],["von", 4],["91", 5],["Jahren", 4]]}

## Example 2:
Input sentence 1: ["Saudis to permit women to compete in Olympics"]
Input sentence 2: ["Saudische Frauen dürfen an den Olympischen Spielen teilnehmen"]
Output: {"sentence1": [["Saudis", 2], ["to", 1], ["permit", 1], ["women", 4], ["to", 0], ["compete", 3], ["in", 5], ["Olympics", 5]], "sentence2": [["Saudische", 2], ["Frauen", 4], ["dürfen", 1], ["an", 5], ["den", 5], ["Olympischen", 5], ["Spielen", 5], ["teilnehmen", 3]]}

## Example 3:
Input sentence 1: ["6.8 quake strikes off Solomon Islands"]
Input sentence 2: ["Beben der Stärke 6,3 vor den Salomon-Inseln: USGS"]
Output: {"sentence1": [["6.8", 1], ["quake", 5], ["strikes", 0], ["off", 5], ["Solomon", 5], ["Islands", 5]], "sentence2": [["Beben", 5], ["der", 3], ["Stärke", 3], ["6", 5], [",", -1], ["3", 0], ["vor", 5], ["den", 5], ["Salomon", 5], ["-", -1], ["Inseln", 5], [":", -1], ["USGS", 0]]}

## Score scale

* **5:** Complete equivalence
* **3-4:** Very similar or closely similar in terms of semantics.
* **1–2** Slightly similar or somewhat similar.
* **0:** No relation (there is no word in the other sentence that is even slightly similar in terms of semantics).
* **-1:** Punctuation

## Other guidelines
* The two texts are given as normal running text. In the output, each text is listed item by item in a fixed order. Punctuation marks, hyphens and apostrophes are separate items, e.g. "Switzerland's" → "Switzerland", "'", "s" and "mid-Winter" → "mid", "-", "Winter".
* The items are fixed; your task is to give every item a score from the scale above, in the order listed.
* Output only the raw JSON, without any additional text.

## Input to Annotate

Sentence 1: {{ sentence1 }}
Sentence 2: {{ sentence2 }}

Respond with the JSON object.
"""

FEWSHOT_PROMPT_FR = """
Annotate a pair of English-French input sentences. The goal is to label every word in each sentence regarding its semantic similarity to the words in the other sentence.

## Example 1:
Input sentence 1: ["Former Nazi death camp guard Demjanjuk dead at 91"]
Input sentence 2: ["John Demjanjuk, gardien des camps de la mort nazis, meurt à l'âge de 91 ans"]
Output: {"sentence1": [["Former", 1],["Nazi", 5],["death", 5],["camp", 5],["guard", 5],["Demjanjuk", 5],["dead", 5],["at", 5],["91", 5]],"sentence2":[["John", 0], ["Demjanjuk", 5], [",", -1], ["gardien", 5], ["des", 5], ["camps", 5], ["de", 5], ["la", 5], ["mort", 5], ["nazis", 3], [",", -1], ["meurt", 5], ["à", 4], ["l", 4], ["'", -1], ["âge", 4], ["de", 4], ["91", 5], ["ans", 4]]}

## Example 2:
Input sentence 1: ["Saudis to permit women to compete in Olympics"]
Input sentence 2: ["Les femmes saoudiennes autorisées à participer aux Jeux olympiques"]
Output: {"sentence1": [["Saudis", 2], ["to", 1], ["permit", 1], ["women", 4], ["to", 0], ["compete", 3], ["in", 5], ["Olympics", 5]], "sentence2": [["Les", 4], ["femmes", 4], ["saoudiennes", 2], ["autorisées", 1], ["à", 3], ["participer", 1], ["aux", 5], ["Jeux", 5], ["olympiques", 5]]}

## Example 3:
Input sentence 1: ["6.8 quake strikes off Solomon Islands"]
Input sentence 2: ["Un séisme de magnitude 6,3 frappe les îles Salomon: USGS"]
Output: {"sentence1": [["6.8", 1], ["quake", 5], ["strikes", 4], ["off", 5], ["Solomon", 5], ["Islands", 5]], "sentence2": [["Un", 5], ["séisme", 5], ["de", 3], ["magnitude", 3], ["6", 5], [",", -1], ["3", 0], ["frappe", 4], ["les", 5], ["îles", 5], ["Salomon", 5], [":", -1], ["USGS", 0]]}

## Score scale

* **5:** Complete equivalence
* **3-4:** Very similar or closely similar in terms of semantics.
* **1–2** Slightly similar or somewhat similar.
* **0:** No relation (there is no word in the other sentence that is even slightly similar in terms of semantics).
* **-1:** Punctuation

## Other guidelines
* The two texts are given as normal running text. In the output, each text is listed item by item in a fixed order. Punctuation marks, hyphens and apostrophes are separate items, e.g. "Switzerland's" → "Switzerland", "'", "s" and "mid-Winter" → "mid", "-", "Winter".
* The items are fixed; your task is to give every item a score from the scale above, in the order listed.
* Output only the raw JSON, without any additional text.

## Input to Annotate

Sentence 1: {{ sentence1 }}
Sentence 2: {{ sentence2 }}

Respond with the JSON object.
"""

FEWSHOT_PROMPT_IT = """
Annotate a pair of English-Italian input sentences. The goal is to label every word in each sentence regarding its semantic similarity to the words in the other sentence.

## Example 1:
Input sentence 1: ["Former Nazi death camp guard Demjanjuk dead at 91"]
Input sentence 2: ["John Demjanjuk, guardia del campo di sterminio nazista, muore a 91 anni"]
Output: {"sentence1": [["Former", 1],["Nazi", 5],["death", 5],["camp", 5],["guard", 5],["Demjanjuk", 5],["dead", 5],["at", 5],["91", 5]],"sentence2": [["John", 0], ["Demjanjuk", 5], [",", -1], ["guardia", 5], ["del", 5], ["campo", 5], ["di", 5], ["sterminio", 5], ["nazista", 3], [",", -1], ["muore", 5], ["a", 4], ["91", 5], ["anni", 4]]}

## Example 2:
Input sentence 1: ["Saudis to permit women to compete in Olympics"]
Input sentence 2: ["Le donne saudite possono gareggiare alle Olimpiadi"]
Output:{"sentence1": [["Saudis", 2], ["to", 1], ["permit", 1], ["women", 4], ["to", 0], ["compete", 3], ["in", 5], ["Olympics", 5]], "sentence2": [["Le", 4], ["donne", 4], ["saudite", 5], ["possono", 1], ["gareggiare", 3], ["alle", 5], ["Olimpiadi", 5]]}

## Example 3:
Input sentence 1: ["6.8 quake strikes off Solomon Islands"]
Input sentence 2: ["Scossa di magnitudo 6,3 al largo delle Isole Salomone: USGS"]
Output: {"sentence1": [["6.8", 1], ["quake", 5], ["strikes", 0], ["off", 5], ["Solomon", 5], ["Islands", 5]], "sentence2": [["Scossa", 5], ["di", 3], ["magnitudo", 3], ["6", 5], [",", -1], ["3", 0], ["al", 5], ["largo", 5], ["delle", 5], ["Isole", 5], ["Salomone", 5], [":", -1], ["USGS", 0]]}

## Score scale

* **5:** Complete equivalence
* **3-4:** Very similar or closely similar in terms of semantics.
* **1–2** Slightly similar or somewhat similar.
* **0:** No relation (there is no word in the other sentence that is even slightly similar in terms of semantics).
* **-1:** Punctuation

## Other guidelines
* The two texts are given as normal running text. In the output, each text is listed item by item in a fixed order. Punctuation marks, hyphens and apostrophes are separate items, e.g. "Switzerland's" → "Switzerland", "'", "s" and "mid-Winter" → "mid", "-", "Winter".
* The items are fixed; your task is to give every item a score from the scale above, in the order listed.
* Output only the raw JSON, without any additional text.

## Input to Annotate

Sentence 1: {{ sentence1 }}
Sentence 2: {{ sentence2 }}

Respond with the JSON object.
"""

FEWSHOT_PROMPT_MIXED = """
Annotate a pair of either English-German, English-French or English-Italian input sentences. The goal is to label every word in each sentence regarding its semantic similarity to the words in the other sentence.

## Example 1:
Input sentence 1: ["Former Nazi death camp guard Demjanjuk dead at 91"]
Input sentence 2: ["John Demjanjuk, verurteilter Nazi-Todeslagerwächter, stirbt im Alter von 91 Jahren"]
Output: {"sentence1": [["Former", 1],["Nazi", 5],["death", 5],["camp", 5],["guard", 5],["Demjanjuk", 5],["dead", 5],["at", 5],["91", 5]],"sentence2": [["John", 0],["Demjanjuk", 5],[",", -1],["verurteilter", 1],["Nazi", 1],["-", -1],["Todeslagerwächter", 5],[",", -1],["stirbt", 5],["im", 4],["Alter", 4],["von", 4],["91", 5],["Jahren", 4]]}


## Example 2:
Input sentence 1: ["Saudis to permit women to compete in Olympics"]
Input sentence 2: ["Les femmes saoudiennes autorisées à participer aux Jeux olympiques"]
Output: {"sentence1": [["Saudis", 2], ["to", 1], ["permit", 1], ["women", 4], ["to", 0], ["compete", 3], ["in", 5], ["Olympics", 5]], "sentence2": [["Les", 4], ["femmes", 4], ["saoudiennes", 2], ["autorisées", 1], ["à", 3], ["participer", 1], ["aux", 5], ["Jeux", 5], ["olympiques", 5]]}


## Example 3:
Input sentence 1: ["6.8 quake strikes off Solomon Islands"]
Input sentence 2: ["Scossa di magnitudo 6,3 al largo delle Isole Salomone: USGS"]
Output: {"sentence1": [["6.8", 1], ["quake", 5], ["strikes", 0], ["off", 5], ["Solomon", 5], ["Islands", 5]], "sentence2": [["Scossa", 5], ["di", 3], ["magnitudo", 3], ["6", 5], [",", -1], ["3", 0], ["al", 5], ["largo", 5], ["delle", 5], ["Isole", 5], ["Salomone", 5], [":", -1], ["USGS", 0]]}

## Score scale

* **5:** Complete equivalence
* **3-4:** Very similar or closely similar in terms of semantics.
* **1–2** Slightly similar or somewhat similar.
* **0:** No relation (there is no word in the other sentence that is even slightly similar in terms of semantics).
* **-1:** Punctuation

## Other guidelines
* The two texts are given as normal running text. In the output, each text is listed item by item in a fixed order. Punctuation marks, hyphens and apostrophes are separate items, e.g. "Switzerland's" → "Switzerland", "'", "s" and "mid-Winter" → "mid", "-", "Winter".
* The items are fixed; your task is to give every item a score from the scale above, in the order listed.
* Output only the raw JSON, without any additional text.

## Input to Annotate

Sentence 1: {{ sentence1 }}
Sentence 2: {{ sentence2 }}

Respond with the JSON object.
"""

In [ ]:
# HELPER FUNCTIONS FOR CONSTANTS
def similarity_to_gold(scores):
    """
    Turn paper prompt values into gold label values for comparison.

    Paper prompt: 5 = equivalent ... 0 = no relation, -1 = punctuation.
    Gold: 0 = equivalent ... 1.0 = completely different, -1 = punctuation.

    Args:
        scores: The paper prompt scores.

    Returns:
        NumPy array with paper prompt scores converted to gold label values.
    """
    scores = np.asarray(scores, dtype=float)
    return np.where(scores == -1, -1.0, np.round((5 - scores) / 5, 1))

# 5->0.0  4->0.2  3->0.4  2->0.6  1->0.8  0->1.0  -1->-1
assert np.array_equal(similarity_to_gold([5, 4, 3, 2, 1, 0, -1]), [0, .2, .4, .6, .8, 1, -1])

In [ ]:
# CONSTANTS

# Environment variables are passed from docker
# Change them in the .env which is passed to docker if they need changing
MODEL_PROVIDER_BASE_URL = os.environ.get("LLM_BASE_URL")
MODEL_PROVIDER_API_KEY = os.environ.get("LLM_API_KEY")
MODEL_NAME = os.environ.get("LLM_NAME")
SPLIT = os.environ.get("SPLIT", "validation")

# Change these params to influence the model predictions
# Results will be written to a file named "preds_{SPLIT}_{MODEL_NAME}_{PROMPT}_temp_{TEMPERATURE}_{TAG}.jsonl"
TEMPERATURE = 0
MAX_TOKENS = 40000
MAX_CONCURRENCY = 64
PROMPT = "paper_fewshot_mixed"
TAG = "batched"

PROMPTS = {
    "paper_fewshot_de": (FEWSHOT_PROMPT_DE, similarity_to_gold),
    "paper_fewshot_fr": (FEWSHOT_PROMPT_FR, similarity_to_gold),
    "paper_fewshot_it": (FEWSHOT_PROMPT_IT, similarity_to_gold),
    "paper_fewshot_mixed": (FEWSHOT_PROMPT_MIXED, similarity_to_gold),
}

SWISSGOV_RSD_DATASET = "ZurichNLP/SwissGov-RSD"
GOLD_VOCAB = np.array([-1, 0, 0.2, 0.4, 0.6, 0.8, 1.0])
SCORE_CHOICES = ["-1", "0", "1", "2", "3", "4", "5"]

RESULTS_DIR = Path.cwd() / "RSD_Validation_Pipeline_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load dataset (validation + test) and inspect both splits
ds = load_dataset(SWISSGOV_RSD_DATASET)
print(ds)

In [ ]:
# Split validation and test sets into their own DataFrame
df_validation = ds["validation"].to_pandas().assign(split="validation")
df_test = ds["test"].to_pandas().assign(split="test")

In [ ]:
# Choose split according to SPLIT constant
df = {"validation": df_validation,
      "test": df_test,
      "full": pd.concat([df_validation, df_test], ignore_index=True)}[SPLIT]
print(f"evaluating on {SPLIT} split: {len(df)} documents")

In [ ]:
# Validate that there is no overlap between validation and test documents
val_pages = set(zip(df_validation["page_en"], df_validation["page_other"]))
test_pages = set(zip(df_test["page_en"], df_test["page_other"]))

page_overlap = val_pages & test_pages
print(f"validation unique source pages: {len(val_pages)}")
print(f"test unique source pages: {len(test_pages)}")
print(f"overlapping source pages: {len(page_overlap)}")

assert not page_overlap, f"{len(page_overlap)} page pairs in both splits, e.g. {next(iter(page_overlap))}"

In [ ]:
# Check that there are no wrong labels

# for validation set
all_labels_validation = np.concatenate(df_validation["labels_a"].tolist() + df_validation["labels_b"].tolist())
print("--------VALIDATION SET--------")
print("label vocabulary:", np.unique(all_labels_validation))
print("num unique values:", len(np.unique(all_labels_validation)))
assert np.array_equal(np.unique(all_labels_validation), GOLD_VOCAB)

# for test set
all_labels_test = np.concatenate(df_test["labels_a"].tolist() + df_test["labels_b"].tolist())
print()
print("--------TEST SET--------")
print("label vocabulary:", np.unique(all_labels_test))
print("num unique values:", len(np.unique(all_labels_test)))
assert np.array_equal(np.unique(all_labels_test), GOLD_VOCAB)

In [ ]:
# Prompt rendering, llguidance grammar and output validation
def render(template, sen_a_untokenized, sen_b_untokenized):
    """
    Renders the complete prompt that is sent to the model.

    Within the prompt templates, the strings `{{ sentence1 }}` and `{{ sentence2 }}`
    are replaced with the running text of document A and B respectively.

    Args:
    template: The correct prompt template from PROMPTS.
    sen_a_untokenized: Text A as running text.
    sen_b_untokenized: Text B as running text.

    Returns:
        The prompt sent to the model.
    """
    return (template
            .replace("{{ sentence1 }}", json.dumps(sen_a_untokenized, ensure_ascii=False))
            .replace("{{ sentence2 }}", json.dumps(sen_b_untokenized, ensure_ascii=False)))

def expected_output(sen_a_tokenized, sen_b_tokenized, scores_a, scores_b):
    """The exact JSON string the grammar allows for the given scores."""
    def side(tokens, scores, key):
        return f'"{key}": [' + ", ".join(f"[{json.dumps(t, ensure_ascii=False)}, {s}]"
                                         for t, s in zip(tokens, scores, strict=True)) + "]"
    return "{" + side(sen_a_tokenized, scores_a, "sentence1") + ", " + side(sen_b_tokenized, scores_b, "sentence2") + "}"

def build_grammar(sen_a_tokenized, sen_b_tokenized):
    """llguidance grammar (Lark-like syntax) for the JSON the paper prompt asks for: every token pinned in order, only the score is free.
    Written as a flat string - chaining guidance select()s nests one level per token and hits the recursion limit on long docs."""
    slot = "\0"
    literals = expected_output(sen_a_tokenized, sen_b_tokenized,
                               [slot] * len(sen_a_tokenized), [slot] * len(sen_b_tokenized)).split(slot)
    return ("%llguidance {}\n\nstart: START\n"
            "START: " + " SCORE ".join(json.dumps(lit) for lit in literals) + "\n"
            "SCORE: " + " | ".join(json.dumps(c) for c in SCORE_CHOICES) + "\n")

def parse_output(sen_a_tokenized, sen_b_tokenized, raw, to_difference):
    """Reject output the grammar doesn't accept (= backend ignored it), then map scores to the gold scale.
    Only the scores are free, so the output is valid iff re-rendering its scores reproduces it exactly."""
    try:
        out = json.loads(raw)
        scores_a = [s for _, s in out["sentence1"]]
        scores_b = [s for _, s in out["sentence2"]]
        valid = (all(str(s) in SCORE_CHOICES for s in scores_a + scores_b)
                 and expected_output(sen_a_tokenized, sen_b_tokenized, scores_a, scores_b) == raw)
    except (TypeError, ValueError, KeyError):
        valid = False
    if not valid:
        raise ValueError("output doesn't match the grammar - was it enforced?")
    pred_a = to_difference(np.array(scores_a, dtype=float))
    pred_b = to_difference(np.array(scores_b, dtype=float))
    assert np.isin(np.r_[pred_a, pred_b], GOLD_VOCAB).all()
    return pred_a, pred_b

In [ ]:
# Model-agnostic predictor interface
class RSDPredictor(ABC):
    """
    Interface for every model that predicts word-level semantic differences.

    The pipeline only uses `name` and `predict()`, so any backend
    (e.g. an OpenAI-compatible endpoint like in the cell below, a local model, etc.)
    can be evaluated by subclassing this class.

    Attributes:
        name: Unique identifier of the run. Used for the cache file.
    """
    name: str

    @abstractmethod
    def predict(
        self,
        sen_a_tokenized: list[str],
        sen_b_tokenized: list[str],
        sen_a_untokenized: str,
        sen_b_untokenized: str
    ) -> tuple[np.ndarray, np.ndarray]:
        """
        Predict the semantic differences between two documents by assigning
        a difference score to each token of those texts.

        Args:
            sen_a_tokenized: Tokens of text A as in the gold labels.
                             The result needs exactly one score per token in this order.
            sen_b_tokenized: Tokens of text B as in the gold labels. Same as above.
            sen_a_untokenized: Text A as running text. This is what the model sees in the prompt.
            sen_b_untokenized: Text B as running text. Same as above.

        Returns:
            (scores_a, scores_b): Float arrays with the same length as sen_a_tokenized and sen_b_tokenized.
                                  Every value is within the GOLD_VOCAB range (-1.0, 0.0, 0.2, 0.4, 0.6, 0.8 or 1.0).
        """
        raise NotImplementedError

In [ ]:
# OpenAI-compatible predictor with llguidance grammar
class OpenAIPredictor(RSDPredictor):
    """
    Predictor for an OpenAI-compatible endpoint with grammar-constrained decoding.

    Every request sends an llguidance grammar via `structured_outputs` that pins all
    tokens of both texts in order so the model can only choose the scores. This quarantees
    exactly one score per token. The endpoint must support these grammars.

    Attributes:
        name: "{SPLIT}:{model_name}:{prompt}:temp_{temperature}:{tag}". During a run, the
        cache file name is derived from it.
    """

    def __init__(
        self,
        model_name=MODEL_NAME,
        prompt=PROMPT,
        base_url=MODEL_PROVIDER_BASE_URL,
        api_key=MODEL_PROVIDER_API_KEY,
        tag=TAG,
        temperature=TEMPERATURE,
        seed=42,
        max_tokens=MAX_TOKENS
    ):
        """
        Set up the client and request parameters.

        The defaults are the constants from the CONSTANTS cell.

        Args:
            model_name: Model name as served by the endpoint.
            prompt: Key in PROMPTS -> selects the prompt template and score conversion
            base_url: Base URL of the endpoint.
            api_key: API key for the endpoint (may not be needed).
            tag: Free label to tell runs apart in the name and cache file.
            temperature: Sampling temperature.
            seed: Sampling seed for reproducibility.
            max_tokens: Maximum number of generated tokens per request.
        """
        self.name = f"{SPLIT}:{model_name}:{prompt}:temp_{temperature}:{tag}"
        self.model_name = model_name
        self.template, self.to_difference = PROMPTS[prompt]
        self.client = OpenAI(base_url=base_url, api_key=api_key, max_retries=0, timeout=900)
        self.params = dict(temperature=temperature, seed=seed, max_tokens=max_tokens)

    def predict(
        self,
        sen_a_tokenized,
        sen_b_tokenized,
        sen_a_untokenized,
        sen_b_untokenized
    ):
        """
        Annotate one document pair.

        Arguments and return value in `RSDPredictor.predict()`.

        Raises:
            RuntimeError: If generation did not finish (i.e. `finish_reason` is not "stop").
            ValueError: If the output doesn't match the grammar.
        """
        resp = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": render(self.template, sen_a_untokenized, sen_b_untokenized)}],
            extra_body={"structured_outputs": {"grammar": build_grammar(sen_a_tokenized, sen_b_tokenized)}},
            **self.params,
        )
        choice = resp.choices[0]
        if choice.finish_reason != "stop":
            raise RuntimeError(f"generation stopped early: {choice.finish_reason}")
        return parse_output(sen_a_tokenized, sen_b_tokenized, choice.message.content, self.to_difference)

In [ ]:
# Instantiate predictor
predictor = OpenAIPredictor()

In [ ]:
# Show currently available models for the predictor
client = predictor.client.with_options(max_retries=0, timeout=5)

for attempt in range(90):
    try:
        models = [m.id for m in client.models.list()]
        print("ready:", models)
        break
    except (APIConnectionError, APIStatusError) as e:
        print(f"not ready ({type(e).__name__}), retrying in 10s")
        time.sleep(10)
else:
    raise RuntimeError(f"no model server reachable at {MODEL_PROVIDER_BASE_URL}")
assert MODEL_NAME in models, f"{MODEL_NAME!r} is not served here; available: {models}"

In [ ]:
# Run over the dataset (cached, resumable, progress bar, concurrent requests)
def read_cache(path):
    rows = []
    for line in (path.read_text().splitlines() if path.exists() else []):
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass   # truncated last line from an interrupted run
    return rows

def run(predictor, frame, max_workers=MAX_CONCURRENCY):
    safe_name = predictor.name.replace(":", "_").replace("/", "_")
    cache = RESULTS_DIR / f"preds_{safe_name}.jsonl"

    done = {r["id"]: r for r in read_cache(cache) if not r["error"]}
    todo = frame[~frame.id.isin(done.keys())]
    n_tokens = todo.text_a.str.split().str.len() + todo.text_b.str.split().str.len()

    print(f"{len(done)} cached, {len(todo)} to run, {n_tokens.sum():,} tokens, {max_workers} concurrent")

    def work(r):
        t0 = perf_counter()
        try:
            a, b = predictor.predict(r.text_a.split(), r.text_b.split(), r.text_a_untokenized, r.text_b_untokenized)
            rec = {"id": r.id, "pred_a": a.tolist(), "pred_b": b.tolist(), "error": None}
        except Exception as e:
            rec = {"id": r.id, "pred_a": None, "pred_b": None, "error": repr(e)}
        rec["seconds"] = round(perf_counter() - t0, 1)
        return r, rec

    n_err = 0
    pool = ThreadPoolExecutor(max_workers)
    try:
        futures = [pool.submit(work, r) for r in todo.itertuples()]
        bar = tqdm(as_completed(futures), total=len(futures), unit="doc", mininterval=30, maxinterval=30)
        with cache.open("a") as f:
            for fut in bar:
                r, rec = fut.result()
                f.write(json.dumps(rec) + "\n"); f.flush()
                if rec["error"]:
                    n_err += 1
                    tqdm.write(f"Error for id {r.id}: {rec['error']}")
                else:
                    done[r.id] = rec
                bar.set_postfix(last=f"{rec['seconds']}s",
                                tokens=len(r.text_a.split()) + len(r.text_b.split()),
                                errors=n_err, refresh=False)
    finally:
        # on interrupt: drop queued docs instead of silently running them all
        pool.shutdown(wait=False, cancel_futures=True)
    return done

preds = run(predictor, df)

In [ ]:
# Evaluate (pooled token-level, gold punctuation excluded)
def evaluate(frame, preds):
    rows = []
    for subset, g in frame.groupby("subset"):
        ok = [(r, preds[r.id]) for r in g.itertuples() if r.id in preds and not preds[r.id]["error"]]
        gold = np.concatenate([np.r_[r.labels_a, r.labels_b] for r, _ in ok]) if ok else np.array([])
        pred = np.concatenate([np.r_[p["pred_a"], p["pred_b"]] for _, p in ok]) if ok else np.array([])
        keep = gold != -1
        gold, pred = gold[keep], pred[keep]
        rows.append({"subset": subset, "docs_ok": len(ok), "docs_failed": len(g) - len(ok),
                     "tokens": len(gold),
                     "spearman": 100 * spearmanr(gold, pred).statistic if len(gold) else np.nan})
    return pd.DataFrame(rows).set_index("subset").round(2)

evaluate(df[df.id.isin(preds)], preds)

In [ ]:
# Literal results, token by token
def results_table(frame, preds):
    """One row per token: id, subset, side, position, token, gold, pred."""
    rows = []
    for r in frame.itertuples():
        p = preds.get(r.id)
        if not p or p["error"]:
            continue
        for side, text, gold, pred in (("a", r.text_a, r.labels_a, p["pred_a"]),
                                       ("b", r.text_b, r.labels_b, p["pred_b"])):
            for i, (tok, g, pr) in enumerate(zip(text.split(), gold, pred)):
                rows.append((r.id, r.subset, r.split, side, i, tok, g, pr))
    return pd.DataFrame(rows, columns=["id", "subset", "split", "side", "pos", "token", "gold", "pred"])


tokens = results_table(df, preds)

# Per-document overview: timing, errors, per-doc Spearman
docs = pd.DataFrame(preds.values()).set_index("id")
docs["n_tokens"] = tokens.groupby("id").size()
docs["spearman"] = tokens[tokens.gold != -1].groupby("id").apply(
    lambda g: spearmanr(g.gold, g.pred).statistic if g.gold.nunique() > 1 and g.pred.nunique() > 1 else np.nan)
docs = docs.drop(columns=["pred_a", "pred_b", "reasoning"], errors="ignore").join(df.set_index("id")[["subset", "split"]])

print(f"{len(tokens):,} tokens from {docs.error.isna().sum()} documents "
      f"({docs.error.notna().sum()} failed), total {docs.seconds.sum()/60:.1f} min")
print("\nConfusion (rows = gold, cols = pred):")
print(pd.crosstab(tokens.gold, tokens.pred))
docs.sort_values("spearman")

In [ ]:
# Compare all models / prompts / splits from their cache files

# Readable names for the table, runs not listed here keep their file name
RUN_LABELS = {}
COMPARISON_FILE = RESULTS_DIR / "comparison.csv"
ALL_DOCS = pd.concat([df_validation, df_test], ignore_index=True)   # ids are unique across splits

def load_cache(path):
    return {r["id"]: r for r in map(json.loads, Path(path).read_text().splitlines())}

rows = []
for path in sorted(RESULTS_DIR.glob("preds_*.jsonl")):
    p = load_cache(path)
    ids = [i for i, r in p.items() if not r["error"]]
    if not ids:
        continue
    frame = ALL_DOCS[ALL_DOCS.id.isin(ids)]
    res = evaluate(frame, p)
    t = results_table(frame, p)
    t = t[t.gold != -1]
    run_id = path.stem.removeprefix("preds_")
    rows.append({"run": RUN_LABELS.get(run_id, run_id),
                 "split": run_id.split("_", 1)[0],
                 "docs": len(ids),
                 "seconds": sum(r.get("seconds", 0) for r in p.values()),
                 "spearman_pooled": 100 * spearmanr(t.gold, t.pred).statistic,
                 **{f"spearman_{s}": v for s, v in res.spearman.items()},
                 "updated": pd.Timestamp.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
                 "file": path.name})


comparison = (pd.DataFrame(rows).set_index("run").round(3)
              .sort_values(["split", "spearman_pooled"], ascending=[True, False]))
comparison.to_csv(COMPARISON_FILE)
print(f"saved to {COMPARISON_FILE}")
comparison